# 🎬 استوديو الاستخراج والتحريك ثلاثي الأبعاد (Multi-Person 3D Mocap & 2.5D Anime Pipeline)
### مُصمّم لمشاهد القتال المتشابكة ومتعددة الشخصيات في الإضاءة المنخفضة (مشهد الممر في Daredevil) مع الحفظ المباشر في GitHub

**المزايا الهندسية في هذا الملف:**
1. **بدون ربط Google Drive (Local + Direct GitHub Push):** يعمل بالكامل على قرص Colab المحلي عالي السرعة ويرفع المخرجات (`.bvh`, `scene_dna.json`, `.blend`, وفيديوهات المعاينة والمقارنة) مباشرة إلى مستودع `h776117073-eng/AI-Creative-Studio`.
2. **معالجة الإضاءة المظلمة والظلال (Low-Light CLAHE Proxy):** رفع تباين الإضاءة والظلال مؤقتاً لتمكين الذكاء الاصطناعي من رؤية أطراف الشخصيات السوداء بدقة في الممر المظلم.
3. **فصل وتتبع الشخصيات المتشابكة (Multi-Person Re-ID Tracking):** استخدام `YOLOv11x-Pose` + `BoT-SORT` لتثبيت هوية كل مقاتل (`ID`) ومنع تداخل الهياكل العظمية أثناء الاشتباك.
4. **الرفع ثلاثي الأبعاد وفصل العمق (3D Lifting + Depth Anything V2):** استخراج عمق الممر (`Z-Axis`) لضمان عدم تداخل المجسمات داخل بعضها في الفضاء ثلاثي الأبعاد.
5. **بناء مشهد Blender آلياً بستايل أنمي 2.5D (Cel-Shading):** إنشاء الممر، الإضاءة الصفراء-الخضراء، وتحريك الشخصيات المتعددة مع رندر المعاينة وحفظ ملف `.blend`.

## ⚙️ الخطوة 1: تجهيز بيئة العمل المحلية (بدون Google Drive) وتثبيت المكتبات

In [ ]:
# @title 1.1 تجهيز مجلدات العمل المحلية وإعدادات مستودع GitHub
import os

GITHUB_OWNER = 'h776117073-eng'
GITHUB_REPO = 'AI-Creative-Studio'
GITHUB_BRANCH = 'main'
OUTPUT_SUBDIR = 'outputs/daredevil_hallway_3d_anime'

BASE_DIR = '/content/Daredevil_Hallway_3D_Anime'
DIRS = {
    'raw': f'{BASE_DIR}/01_raw_video',
    'proxy': f'{BASE_DIR}/02_enhanced_proxy',
    'tracking': f'{BASE_DIR}/03_multiperson_tracking',
    'depth': f'{BASE_DIR}/04_depth_and_camera',
    'bvh': f'{BASE_DIR}/05_bvh_mocap',
    'blender': f'{BASE_DIR}/06_blender_project',
    'render': f'{BASE_DIR}/07_final_renders'
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

print(f'✅ تم تجهيز مجلدات العمل المحلية (بدون Google Drive) في: {BASE_DIR}')
print(f'📦 المستودع المستهدف للحفظ: {GITHUB_OWNER}/{GITHUB_REPO} -> {OUTPUT_SUBDIR}')

In [ ]:
# @title 1.2 تثبيت حزم الرؤية الحاسوبية، التتبع، العمق، الصوت، و yt-dlp
!pip install -q -U yt-dlp ultralytics lapx scipy librosa transformers timm accelerate einops opencv-python-headless requests
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🚀 الجهاز النشط: {device.upper()}', torch.cuda.get_device_name(0) if device == 'cuda' else '')

## 📥 الخطوة 2: تحميل المشهد وقص المقطع المستهدف بدقة

In [ ]:
# @title 2.1 تحميل الفيديو من يوتيوب وتحديد بداية ونهاية المقطع
import subprocess, json

YOUTUBE_URL = 'https://youtu.be/B66feInucFY'
START_TIME = '00:00:50'
END_TIME = '00:01:15'
PROCESS_FULL_VIDEO = False

RAW_FULL = f"{DIRS['raw']}/full_source.mp4"
CLIP_PATH = f"{DIRS['raw']}/active_clip.mp4"
AUDIO_PATH = f"{DIRS['raw']}/active_audio.wav"

if not os.path.exists(RAW_FULL):
    print('⏳ جاري تحميل الفيديو بأعلى دقة (1080p)...')
    !yt-dlp -f 'bestvideo[height<=1080][ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best' --merge-output-format mp4 -o '{RAW_FULL}' '{YOUTUBE_URL}'
else:
    print('✅ الفيديو الأصلي موجود مسبقاً.')

if PROCESS_FULL_VIDEO:
    !ffmpeg -y -i '{RAW_FULL}' -c copy '{CLIP_PATH}' -loglevel error
else:
    !ffmpeg -y -ss {START_TIME} -to {END_TIME} -i '{RAW_FULL}' -c:v libx264 -crf 16 -preset fast -c:a aac '{CLIP_PATH}' -loglevel error

!ffmpeg -y -i '{CLIP_PATH}' -vn -acodec pcm_s16le -ar 22050 -ac 1 '{AUDIO_PATH}' -loglevel error

probe_cmd = ['ffprobe', '-v', 'quiet', '-print_format', 'json', '-show_streams', '-show_format', CLIP_PATH]
meta = json.loads(subprocess.check_output(probe_cmd))
v_stream = [s for s in meta['streams'] if s['codec_type'] == 'video'][0]
fps_num, fps_den = map(int, v_stream['r_frame_rate'].split('/'))
VIDEO_FPS = fps_num / fps_den
WIDTH, HEIGHT = int(v_stream['width']), int(v_stream['height'])
DURATION = float(meta['format']['duration'])
print(f'🎬 المقطع جاهز: {WIDTH}x{HEIGHT} | {VIDEO_FPS:.2f} FPS | المدة: {DURATION:.2f} ثانية')

## 💡 الخطوة 3: معالجة الإضاءة المظلمة (Low-Light CLAHE Proxy)

In [ ]:
# @title 3.1 إنشاء فيديو التحليل المحسّن (CLAHE + Gamma Lift) واستخراج بصمة الإضاءة
import cv2
import numpy as np
from tqdm.auto import tqdm

PROXY_PATH = f"{DIRS['proxy']}/enhanced_analysis_proxy.mp4"
GAMMA = 1.75
CLAHE_CLIP = 3.5

cap = cv2.VideoCapture(CLIP_PATH)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(PROXY_PATH, fourcc, VIDEO_FPS, (WIDTH, HEIGHT))

inv_gamma = 1.0 / GAMMA
lut = np.array([((i / 255.0) ** inv_gamma) * 255 for i in range(256)]).astype('uint8')
clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=(8, 8))

avg_bgr_samples = []

for idx in tqdm(range(total_frames), desc='تحسين الإضاءة للتحليل'):
    ret, frame = cap.read()
    if not ret:
        break
    if idx % 15 == 0:
        avg_bgr_samples.append(frame.mean(axis=(0, 1)))

    bright = cv2.LUT(frame, lut)
    lab = cv2.cvtColor(bright, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l_eq = clahe.apply(l)
    enhanced = cv2.cvtColor(cv2.merge((l_eq, a, b)), cv2.COLOR_LAB2BGR)
    out.write(enhanced)

cap.release()
out.release()

mean_bgr = np.mean(avg_bgr_samples, axis=0)
lighting_profile = {
    'dominant_rgb_0_1': [round(float(mean_bgr[2]/255.0), 4), round(float(mean_bgr[1]/255.0), 4), round(float(mean_bgr[0]/255.0), 4)],
    'ambient_hex': '#{:02x}{:02x}{:02x}'.format(int(mean_bgr[2]), int(mean_bgr[1]), int(mean_bgr[0])),
    'style': 'Low-Key Industrial Fluorescent Corridor'
}
with open(f"{DIRS['depth']}/lighting_profile.json", 'w') as f:
    json.dump(lighting_profile, f, indent=2)

print(f'✅ تم إنتاج نسخة التحليل بنجاح | لون الإضاءة السائد في المشهد: {lighting_profile["ambient_hex"]}')

## 🧠 الخطوة 4: استخراج الهياكل العظمية المتعددة + العمق (Depth) + مسار الكاميرا

In [ ]:
# @title 4.1 تشغيل التتبع متعدد الشخصيات + خرائط العمق + حركة الكاميرا
from ultralytics import YOLO
from transformers import pipeline as hf_pipeline
from PIL import Image
import librosa

pose_model = YOLO('yolo11x-pose.pt')
depth_pipe = hf_pipeline(task='depth-estimation', model='depth-anything/Depth-Anything-V2-Small-hf', device=0 if device=='cuda' else -1)

y_audio, sr = librosa.load(AUDIO_PATH, sr=22050)
onset_env = librosa.onset.onset_strength(y=y_audio, sr=sr)
onset_frames = librosa.onset.onset_detect(onset_envelope=onset_env, sr=sr, backtrack=False)
impact_timestamps = [round(float(t), 3) for t in librosa.frames_to_time(onset_frames, sr=sr)]

cap = cv2.VideoCapture(PROXY_PATH)
tracks_data = {}
camera_motion = []
prev_gray = None

DEPTH_STRIDE = 2
last_depth_meters = None

debug_out = cv2.VideoWriter(f"{DIRS['tracking']}/tracking_debug.mp4", cv2.VideoWriter_fourcc(*'mp4v'), VIDEO_FPS, (WIDTH, HEIGHT))

for frame_idx in tqdm(range(total_frames), desc='استخراج الشخصيات والعمق والكاميرا'):
    ret, frame = cap.read()
    if not ret:
        break

    results = pose_model.track(
        frame, persist=True, tracker='botsort.yaml',
        conf=0.25, iou=0.5, imgsz=960, verbose=False
    )[0]

    if frame_idx % DEPTH_STRIDE == 0 or last_depth_meters is None:
        pil_img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        depth_res = depth_pipe(pil_img)['depth']
        depth_arr = np.array(depth_res, dtype=np.float32)
        d_min, d_max = depth_arr.min(), depth_arr.max()
        depth_norm = 1.0 - ((depth_arr - d_min) / (d_max - d_min + 1e-6))
        last_depth_meters = 1.5 + depth_norm * 10.5

    bg_mask = np.ones((HEIGHT, WIDTH), dtype=np.uint8) * 255

    if results.boxes is not None and results.boxes.id is not None and results.keypoints is not None:
        ids = results.boxes.id.int().cpu().tolist()
        boxes = results.boxes.xyxy.cpu().numpy()
        kpts_xy = results.keypoints.xy.cpu().numpy()
        kpts_conf = results.keypoints.conf.cpu().numpy()

        for tid, box, xy, cf in zip(ids, boxes, kpts_xy, kpts_conf):
            x1, y1, x2, y2 = map(int, box)
            cv2.rectangle(bg_mask, (max(0, x1-15), max(0, y1-15)), (min(WIDTH, x2+15), min(HEIGHT, y2+15)), 0, -1)

            hip_pts = []
            for h_idx in [11, 12]:
                if cf[h_idx] > 0.25:
                    px = int(np.clip(xy[h_idx, 0], 0, WIDTH - 1))
                    py = int(np.clip(xy[h_idx, 1], 0, HEIGHT - 1))
                    hip_pts.append(float(last_depth_meters[py, px]))
            if hip_pts:
                root_z = float(np.median(hip_pts))
            else:
                cx = int(np.clip((x1 + x2) / 2, 0, WIDTH - 1))
                cy = int(np.clip((y1 + y2) / 2, 0, HEIGHT - 1))
                root_z = float(last_depth_meters[cy, cx])

            focal_px = WIDTH * 0.85
            joints_3d = []
            for j in range(17):
                u, v = float(xy[j, 0]), float(xy[j, 1])
                c = float(cf[j])
                px = int(np.clip(u, 0, WIDTH - 1))
                py = int(np.clip(v, 0, HEIGHT - 1))
                local_z = float(last_depth_meters[py, px])
                z = root_z + np.clip(local_z - root_z, -0.45, 0.45)
                x = (u - WIDTH / 2.0) * z / focal_px
                y = -(v - HEIGHT / 2.0) * z / focal_px
                joints_3d.append([round(x, 4), round(y, 4), round(z, 4), round(c, 3)])

            if tid not in tracks_data:
                tracks_data[tid] = {'frames': {}, 'avg_box_area': []}
            tracks_data[tid]['frames'][frame_idx] = joints_3d
            tracks_data[tid]['avg_box_area'].append(float((x2 - x1) * (y2 - y1)))

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray_small = cv2.resize(gray, (320, 180))
    mask_small = cv2.resize(bg_mask, (320, 180), interpolation=cv2.INTER_NEAREST)
    if prev_gray is not None:
        flow = cv2.calcOpticalFlowFarneback(prev_gray, gray_small, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        valid = mask_small > 128
        if np.sum(valid) > 200:
            dx = float(np.median(flow[valid, 0]))
            dy = float(np.median(flow[valid, 1]))
            mag = float(np.mean(np.hypot(flow[valid, 0], flow[valid, 1])))
        else:
            dx, dy, mag = 0.0, 0.0, 0.0
        camera_motion.append({'frame': frame_idx, 'dx': round(dx, 4), 'dy': round(dy, 4), 'energy': round(mag, 4)})
    else:
        camera_motion.append({'frame': 0, 'dx': 0.0, 'dy': 0.0, 'energy': 0.0})
    prev_gray = gray_small

    debug_out.write(results.plot())

cap.release()
debug_out.release()

MIN_TRACK_FRAMES = 15
clean_tracks = {tid: data for tid, data in tracks_data.items() if len(data['frames']) >= MIN_TRACK_FRAMES}
print(f'✅ تم استخراج {len(clean_tracks)} شخصيات رئيسية مستقلة من المشهد!')
for tid, data in sorted(clean_tracks.items(), key=lambda x: len(x[1]['frames']), reverse=True):
    f_list = sorted(data['frames'].keys())
    print(f'   👤 شخصية ID={tid:02d} | عدد الإطارات: {len(f_list)} | من الإطار {f_list[0]} إلى {f_list[-1]}')

## 🦴 الخطوة 5: تنعيم الحركة (Occlusion Interpolation) وتصدير ملفات `BVH` + `Scene DNA`

In [ ]:
# @title 5.1 تنعيم المسارات ثلاثية الأبعاد وحفظ ملفات BVH و JSON
from scipy.signal import savgol_filter
from scipy.interpolate import interp1d

smoothed_scene = {
    'video_meta': {'width': WIDTH, 'height': HEIGHT, 'fps': VIDEO_FPS, 'total_frames': total_frames},
    'lighting': lighting_profile,
    'impact_timestamps': impact_timestamps,
    'camera_motion': camera_motion,
    'characters': {}
}

def smooth_track_3d(frames_dict):
    sorted_f = sorted(frames_dict.keys())
    f_start, f_end = sorted_f[0], sorted_f[-1]
    full_range = np.arange(f_start, f_end + 1)
    arr = np.zeros((len(full_range), 17, 3), dtype=np.float32)

    for j in range(17):
        valid_f, valid_xyz = [], []
        for f in sorted_f:
            pt = frames_dict[f][j]
            if pt[3] > 0.25:
                valid_f.append(f)
                valid_xyz.append(pt[:3])
        if len(valid_f) >= 2:
            valid_xyz = np.array(valid_xyz)
            for axis in range(3):
                fn = interp1d(valid_f, valid_xyz[:, axis], kind='linear', fill_value='extrapolate')
                sig = fn(full_range)
                if len(sig) >= 9:
                    win = min(11, len(sig) if len(sig) % 2 == 1 else len(sig) - 1)
                    sig = savgol_filter(sig, window_length=win, polyorder=2)
                arr[:, j, axis] = sig
        elif len(valid_f) == 1:
            arr[:, j, :] = valid_xyz[0]
    return f_start, f_end, arr

BVH_SKELETON_TEMPLATE = '''HIERARCHY
ROOT Hips
{
  OFFSET 0.00 0.00 0.00
  CHANNELS 6 Xposition Yposition Zposition Zrotation Xrotation Yrotation
  JOINT Spine
  {
    OFFSET 0.00 0.35 0.00
    CHANNELS 3 Zrotation Xrotation Yrotation
    JOINT Head
    {
      OFFSET 0.00 0.35 0.00
      CHANNELS 3 Zrotation Xrotation Yrotation
      End Site
      {
        OFFSET 0.00 0.20 0.00
      }
    }
    JOINT LeftArm
    {
      OFFSET 0.18 0.25 0.00
      CHANNELS 3 Zrotation Xrotation Yrotation
      JOINT LeftForeArm
      {
        OFFSET 0.28 0.00 0.00
        CHANNELS 3 Zrotation Xrotation Yrotation
        End Site
        {
          OFFSET 0.25 0.00 0.00
        }
      }
    }
    JOINT RightArm
    {
      OFFSET -0.18 0.25 0.00
      CHANNELS 3 Zrotation Xrotation Yrotation
      JOINT RightForeArm
      {
        OFFSET -0.28 0.00 0.00
        CHANNELS 3 Zrotation Xrotation Yrotation
        End Site
        {
          OFFSET -0.25 0.00 0.00
        }
      }
    }
  }
  JOINT LeftUpLeg
  {
    OFFSET 0.10 -0.05 0.00
    CHANNELS 3 Zrotation Xrotation Yrotation
    JOINT LeftLeg
    {
      OFFSET 0.00 -0.42 0.00
      CHANNELS 3 Zrotation Xrotation Yrotation
      End Site
      {
        OFFSET 0.00 -0.40 0.00
      }
    }
  }
  JOINT RightUpLeg
  {
    OFFSET -0.10 -0.05 0.00
    CHANNELS 3 Zrotation Xrotation Yrotation
    JOINT RightLeg
    {
      OFFSET 0.00 -0.42 0.00
      CHANNELS 3 Zrotation Xrotation Yrotation
      End Site
      {
        OFFSET 0.00 -0.40 0.00
      }
    }
  }
}
MOTION'''

def vec_to_euler(v):
    norm = np.linalg.norm(v) + 1e-6
    vn = v / norm
    rx = np.degrees(np.arcsin(np.clip(-vn[1], -1, 1)))
    ry = np.degrees(np.arctan2(vn[0], vn[2]))
    rz = np.degrees(np.arctan2(vn[1], vn[0])) * 0.3
    return rz, rx, ry

def export_simple_bvh(filepath, poses_arr, fps):
    n_frames = poses_arr.shape[0]
    lines = []
    for f in range(n_frames):
        p = poses_arr[f]
        hips = (p[11] + p[12]) / 2.0
        shoulders = (p[5] + p[6]) / 2.0
        spine_v = shoulders - hips
        head_v = p[0] - shoulders
        l_arm = p[7] - p[5]; l_fore = p[9] - p[7]
        r_arm = p[8] - p[6]; r_fore = p[10] - p[8]
        l_leg = p[13] - p[11]; l_shin = p[15] - p[13]
        r_leg = p[14] - p[12]; r_shin = p[16] - p[14]

        vals = [hips[0]*100, hips[1]*100, hips[2]*100, 0.0, 0.0, 0.0]
        for v in [spine_v, head_v, l_arm, l_fore, r_arm, r_fore, l_leg, l_shin, r_leg, r_shin]:
            vals.extend(vec_to_euler(v))
        lines.append(' '.join(f'{x:.3f}' for x in vals))

    header = f'{BVH_SKELETON_TEMPLATE}\nFrames: {n_frames}\nFrame Time: {1.0/fps:.6f}\n'
    with open(filepath, 'w') as fp:
        fp.write(header + '\n'.join(lines))

sorted_tids = sorted(clean_tracks.keys(), key=lambda t: len(clean_tracks[t]['frames']), reverse=True)
for rank, tid in enumerate(sorted_tids, start=1):
    f_start, f_end, arr = smooth_track_3d(clean_tracks[tid]['frames'])
    role = 'Hero_Daredevil_BlackSuit' if rank == 1 else f'Mobster_{rank-1:02d}'
    bvh_file = f"{DIRS['bvh']}/char_{rank:02d}_{role}.bvh"
    export_simple_bvh(bvh_file, arr, VIDEO_FPS)
    smoothed_scene['characters'][f'char_{rank:02d}'] = {
        'track_id': int(tid),
        'role': role,
        'start_frame': int(f_start),
        'end_frame': int(f_end),
        'bvh_path': bvh_file,
        'joints_3d': np.round(arr, 4).tolist()
    }

SCENE_DNA_PATH = f"{DIRS['depth']}/scene_dna.json"
with open(SCENE_DNA_PATH, 'w') as f:
    json.dump(smoothed_scene, f)

print(f'✅ تم حفظ ملف Scene DNA الكامل في: {SCENE_DNA_PATH}')
print(f'✅ تم تصدير {len(sorted_tids)} ملفات BVH جاهزة للتحريك في: {DIRS["bvh"]}')

## 🎨 الخطوة 6: بناء مشهد Blender ثلاثي الأبعاد بستايل أنمي (2.5D Cel-Shading) ورندره تلقائياً

In [ ]:
# @title 6.1 تثبيت Blender 4.2 LTS (Headless) داخل Google Colab
import shutil
if not shutil.which('blender'):
    print('⏳ جاري تحميل وتثبيت Blender 4.2 LTS...')
    !wget -q https://download.blender.org/release/Blender4.2/blender-4.2.3-linux-x64.tar.xz -O /tmp/blender.tar.xz
    !tar -xf /tmp/blender.tar.xz -C /opt/
    !ln -sf /opt/blender-4.2.3-linux-x64/blender /usr/local/bin/blender
    !rm /tmp/blender.tar.xz
!blender --version | head -n 2

In [ ]:
# @title 6.2 توليد سكريبت Blender الاحترافي وتشغيله لبناء المشهد والرندر
ANIME_STEPPED_12FPS = True
RENDER_PREVIEW_VIDEO = True
RENDER_MAX_FRAMES = 96

BLENDER_SCRIPT_PATH = f"{DIRS['blender']}/build_hallway_anime_scene.py"
BLEND_OUT_PATH = f"{DIRS['blender']}/Daredevil_Hallway_Anime.blend"
RENDER_OUT_PATH = f"{DIRS['render']}/hallway_anime_3d_preview.mp4"

blender_script = f'''import bpy, json, math, os
from mathutils import Vector

SCENE_DNA = r"{SCENE_DNA_PATH}"
BLEND_OUT = r"{BLEND_OUT_PATH}"
RENDER_OUT = r"{RENDER_OUT_PATH}"
STEPPED_12FPS = {ANIME_STEPPED_12FPS}
DO_RENDER = {RENDER_PREVIEW_VIDEO}
MAX_RENDER_F = {RENDER_MAX_FRAMES}

with open(SCENE_DNA, 'r') as f:
    dna = json.load(f)

bpy.ops.wm.read_factory_settings(use_empty=True)
scene = bpy.context.scene
fps = int(round(dna['video_meta']['fps']))
total_f = dna['video_meta']['total_frames']
scene.render.fps = fps
scene.frame_start = 0
scene.frame_end = total_f - 1

scene.render.engine = 'CYCLES'
scene.cycles.device = 'CPU'
scene.cycles.samples = 12
scene.cycles.use_denoising = True
scene.render.resolution_x = 1280
scene.render.resolution_y = 720
scene.render.image_settings.file_format = 'FFMPEG'
scene.render.ffmpeg.format = 'MPEG4'
scene.render.ffmpeg.codec = 'H264'
scene.render.filepath = RENDER_OUT

scene.render.use_freestyle = True
scene.view_layers[0].freestyle_settings.crease_angle = math.radians(135)

def make_toon_mat(name, base_rgba, emission_strength=0.0):
    mat = bpy.data.materials.new(name=name)
    mat.use_nodes = True
    nodes = mat.node_tree.nodes
    links = mat.node_tree.links
    nodes.clear()
    out = nodes.new('ShaderNodeOutputMaterial')
    if emission_strength > 0:
        em = nodes.new('ShaderNodeEmission')
        em.inputs['Color'].default_value = base_rgba
        em.inputs['Strength'].default_value = emission_strength
        links.new(em.outputs['Emission'], out.inputs['Surface'])
        return mat
    toon = nodes.new('ShaderNodeBsdfToon')
    toon.inputs['Color'].default_value = base_rgba
    toon.inputs['Size'].default_value = 0.65
    toon.inputs['Smooth'].default_value = 0.05
    links.new(toon.outputs['BSDF'], out.inputs['Surface'])
    return mat

wall_mat = make_toon_mat('Hallway_Wall', (0.28, 0.32, 0.14, 1))
floor_mat = make_toon_mat('Hallway_Floor', (0.18, 0.20, 0.10, 1))
light_tube_mat = make_toon_mat('Fluorescent_Light', (0.95, 0.92, 0.45, 1), emission_strength=8.0)

def add_box(name, loc, scale, mat):
    bpy.ops.mesh.primitive_cube_add(location=loc)
    obj = bpy.context.active_object
    obj.name = name
    obj.scale = scale
    obj.data.materials.append(mat)
    return obj

add_box('Floor', (0, 6.0, -1.05), (1.3, 6.5, 0.05), floor_mat)
add_box('Ceiling', (0, 6.0, 1.55), (1.3, 6.5, 0.05), wall_mat)
add_box('Wall_Left', (-1.25, 6.0, 0.25), (0.08, 6.5, 1.35), wall_mat)
add_box('Wall_Right', (1.25, 6.0, 0.25), (0.08, 6.5, 1.35), wall_mat)
add_box('Back_Door_Wall', (0, 12.2, 0.25), (1.3, 0.1, 1.35), wall_mat)

for y_pos in [2.5, 6.5, 10.5]:
    add_box(f'LightBox_{{y_pos}}', (0, y_pos, 1.48), (0.45, 0.12, 0.03), light_tube_mat)
    l_data = bpy.data.lights.new(name=f'Lamp_{{y_pos}}', type='AREA')
    l_data.energy = 140
    l_data.color = (0.92, 0.90, 0.42)
    l_data.size = 0.8
    l_obj = bpy.data.objects.new(name=f'Lamp_{{y_pos}}', object_data=l_data)
    l_obj.location = (0, y_pos, 1.38)
    scene.collection.objects.link(l_obj)

cam_data = bpy.data.cameras.new('Main_Cam')
cam_data.lens = 32.0
cam_obj = bpy.data.objects.new('Main_Cam', cam_data)
cam_obj.location = (0.0, 0.0, 0.25)
cam_obj.rotation_euler = (math.radians(90), 0.0, 0.0)
scene.collection.objects.link(cam_obj)
scene.camera = cam_obj

cum_pan, cum_tilt = 0.0, 0.0
for cm in dna['camera_motion']:
    f = cm['frame']
    cum_pan = cum_pan * 0.92 - cm['dx'] * 0.0025
    cum_tilt = cum_tilt * 0.92 - cm['dy'] * 0.0025
    cam_obj.rotation_euler = (math.radians(90) + cum_tilt, 0.0, cum_pan)
    cam_obj.keyframe_insert(data_path='rotation_euler', frame=f)

LIMBS = [
    (5, 6, 0.065), (11, 12, 0.065), (5, 11, 0.075), (6, 12, 0.075),
    (5, 7, 0.048), (7, 9, 0.042), (6, 8, 0.048), (8, 10, 0.042),
    (11, 13, 0.058), (13, 15, 0.048), (12, 14, 0.058), (14, 16, 0.048)
]

hero_suit = make_toon_mat('Daredevil_Suit', (0.06, 0.06, 0.08, 1))
hero_wrap = make_toon_mat('Hand_Wraps', (0.88, 0.88, 0.82, 1))
mob_colors = [
    (0.18, 0.22, 0.28, 1),
    (0.30, 0.20, 0.16, 1),
    (0.20, 0.26, 0.20, 1),
    (0.32, 0.30, 0.24, 1),
]

def orient_limb(obj, p1, p2):
    v1 = Vector((p1[0], p1[2], p1[1]))
    v2 = Vector((p2[0], p2[2], p2[1]))
    diff = v2 - v1
    length = max(diff.length, 0.02)
    obj.location = (v1 + v2) / 2.0
    obj.rotation_mode = 'QUATERNION'
    obj.rotation_quaternion = diff.to_track_quat('Z', 'Y')
    obj.scale.z = length / 2.0

for c_idx, (c_key, c_data) in enumerate(dna['characters'].items()):
    is_hero = (c_idx == 0)
    body_mat = hero_suit if is_hero else make_toon_mat(f'MobMat_{{c_key}}', mob_colors[(c_idx - 1) % len(mob_colors)])

    bpy.ops.mesh.primitive_uv_sphere_add(radius=0.11)
    head = bpy.context.active_object
    head.name = f'{{c_key}}_Head'
    head.data.materials.append(body_mat)

    limb_objs = []
    for l_idx, (j1, j2, rad) in enumerate(LIMBS):
        bpy.ops.mesh.primitive_cylinder_add(radius=rad, depth=2.0, vertices=12)
        cyl = bpy.context.active_object
        cyl.name = f'{{c_key}}_Limb_{{l_idx}}'
        cyl.data.materials.append(hero_wrap if (is_hero and l_idx in [5, 7]) else body_mat)
        limb_objs.append((cyl, j1, j2))

    f_start = c_data['start_frame']
    joints_seq = c_data['joints_3d']
    step = 2 if STEPPED_12FPS else 1

    for local_f in range(0, len(joints_seq), step):
        f = f_start + local_f
        pts = joints_seq[local_f]
        nose = pts[0]
        head.location = (nose[0], nose[2], nose[1])
        head.keyframe_insert(data_path='location', frame=f)
        for cyl, j1, j2 in limb_objs:
            orient_limb(cyl, pts[j1], pts[j2])
            cyl.keyframe_insert(data_path='location', frame=f)
            cyl.keyframe_insert(data_path='rotation_quaternion', frame=f)
            cyl.keyframe_insert(data_path='scale', frame=f)

if STEPPED_12FPS:
    for action in bpy.data.actions:
        for fcurve in action.fcurves:
            for kp in fcurve.keyframe_points:
                kp.interpolation = 'CONSTANT'

bpy.ops.wm.save_as_mainfile(filepath=BLEND_OUT)
print(f'✅ تم حفظ مشروع Blender الكامل في: {{BLEND_OUT}}')

if DO_RENDER:
    scene.frame_end = min(scene.frame_end, MAX_RENDER_F)
    bpy.ops.render.render(animation=True)
    print(f'🎬 تم رندر فيديو الأنمي ثلاثي الأبعاد في: {{RENDER_OUT}}')
'''

with open(BLENDER_SCRIPT_PATH, 'w', encoding='utf-8') as f:
    f.write(blender_script)

print('🚀 جاري تشغيل Blender لبناء المشهد ورندره...')
!blender --background --python '{BLENDER_SCRIPT_PATH}'

## 🎥 الخطوة 7: إنشاء فيديو المقارنة وحفظ جميع المخرجات مباشرة في GitHub (`h776117073-eng/AI-Creative-Studio`)

In [ ]:
# @title 7.1 دمج فيديو المقارنة ورفع المخرجات إلى GitHub تلقائياً
import base64, glob, requests

COMPARE_OUT = f"{DIRS['render']}/comparison_original_vs_tracking.mp4"
TRACK_DEBUG = f"{DIRS['tracking']}/tracking_debug.mp4"

!ffmpeg -y -i '{CLIP_PATH}' -i '{TRACK_DEBUG}' -filter_complex '[0:v]scale=640:360[v0];[1:v]scale=640:360[v1];[v0][v1]hstack=inputs=2[out]' -map '[out]' -map 0:a? -c:v libx264 -crf 22 -shortest '{COMPARE_OUT}' -loglevel error

GH_TOKEN = os.environ.get('GH_TOKEN', '').strip() or os.environ.get('GITHUB_TOKEN', '').strip()
if not GH_TOKEN:
    raise RuntimeError('GH_TOKEN is missing in Colab runtime environment!')

headers = {
    'Authorization': f'token {GH_TOKEN}',
    'Accept': 'application/vnd.github+json'
}

def upload_file_to_github(local_path, repo_path):
    if not os.path.exists(local_path):
        print(f'⚠️ ملف غير موجود: {local_path}')
        return
    size_mb = os.path.getsize(local_path) / (1024 * 1024)
    if size_mb > 45:
        print(f'⚠️ تخطي {repo_path} لأن حجمه {size_mb:.1f}MB يتجاوز حد API.')
        return
    with open(local_path, 'rb') as fp:
        b64_content = base64.b64encode(fp.read()).decode('utf-8')
    url = f'https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPO}/contents/{repo_path}'
    r_get = requests.get(url, headers=headers, params={'ref': GITHUB_BRANCH})
    sha = r_get.json().get('sha') if r_get.status_code == 200 else None
    payload = {
        'message': f'feat(mocap): save {os.path.basename(repo_path)} from Colab pipeline',
        'content': b64_content,
        'branch': GITHUB_BRANCH
    }
    if sha:
        payload['sha'] = sha
    r_put = requests.put(url, headers=headers, json=payload)
    if r_put.status_code in (200, 201):
        print(f'✅ تم رفع الملف بنجاح إلى GitHub: {repo_path} ({size_mb:.2f} MB)')
    else:
        print(f'❌ فشل رفع {repo_path}: {r_put.status_code} {r_put.text[:200]}')

# 1. رفع ملفات التحليل وملفات الحركة BVH
upload_file_to_github(f"{DIRS['depth']}/lighting_profile.json", f"{OUTPUT_SUBDIR}/lighting_profile.json")
upload_file_to_github(SCENE_DNA_PATH, f"{OUTPUT_SUBDIR}/scene_dna.json")
for bvh_f in sorted(glob.glob(f"{DIRS['bvh']}/*.bvh")):
    upload_file_to_github(bvh_f, f"{OUTPUT_SUBDIR}/bvh/{os.path.basename(bvh_f)}")

# 2. رفع سكريبت ومشروع Blender
upload_file_to_github(BLENDER_SCRIPT_PATH, f"{OUTPUT_SUBDIR}/blender/build_hallway_anime_scene.py")
upload_file_to_github(BLEND_OUT_PATH, f"{OUTPUT_SUBDIR}/blender/Daredevil_Hallway_Anime.blend")

# 3. رفع فيديوهات المعاينة والمقارنة
upload_file_to_github(COMPARE_OUT, f"{OUTPUT_SUBDIR}/renders/comparison_original_vs_tracking.mp4")
upload_file_to_github(RENDER_OUT_PATH, f"{OUTPUT_SUBDIR}/renders/hallway_anime_3d_preview.mp4")
print('🎉 اكتمل التنفيذ وحفظ جميع المخرجات في GitHub بنجاح!')